# 20 – Symbolerkennung live: Das Modell an der Prüfstation einsetzen

**Lernsituation „KI-Sichtprüfstation“ · Doppelstunden 9 – 10**

### Lernziele
Du kannst …
- ein trainiertes Modell auf dem Raspberry Pi mit TensorFlow Lite ausführen,
- sicherstellen, dass die Bildvorbereitung im Betrieb exakt der im Training entspricht,
- Modelle nach Genauigkeit, Rechenzeit und Speicherbedarf vergleichen und auswählen,
- Konfidenzschwelle, Taktzeit, Signalausgabe und Protokollierung einer Prüfstation umsetzen,
- die Pipeline auf eine eigene industrielle Aufgabenstellung übertragen.

| Teil | Doppelstunde | Inhalt |
|---|:--:|---|
| A | 9 | Modell auswählen, Live-Erkennung, Taktzeit, Ampel, Protokoll · 🧪 Teilzielkontrolle 4 |
| B | 10 | Projekt: eigene Klassen aus der Fertigung – aufnehmen, trainieren, einsetzen, präsentieren |

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Grundlagen lesen / interaktiv erkunden |
| ✏️ | Aufgabe: Code **selbst schreiben** (Zellen mit `# TODO`) |
| 🔍 | Experiment / Beobachtung – Ergebnisse notieren |
| 🧪 | Teilzielkontrolle |
| 🏭 | Bezug zur industriellen Fertigung |
| 🍓 | Hinweis für den Raspberry Pi |

In [ ]:
# =============================================================================
# Vorbereitung: Hilfsmodul finden (klappt im Hauptordner und in loesungen/)
# =============================================================================
import sys
from pathlib import Path

ORDNER = Path.cwd() if (Path.cwd() / "ki_hilfen.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ORDNER))

import numpy as np
import cv2
import matplotlib.pyplot as plt
import ki_hilfen as kh

import json
import time
import pandas as pd
from datetime import datetime

---
## Teil A – Vom Training zum Betrieb (Doppelstunde 9)

### 📖 Was im Betrieb gleich bleiben muss
Das Netz kennt nur die Welt seiner Trainingsdaten. Jede Abweichung in der Vorverarbeitung führt zu falschen Ergebnissen – ohne Fehlermeldung!

| Punkt | Wie wir es sicherstellen |
|---|---|
| Bildgröße, Farbe/Grau | stehen in `modelle/<name>_info.json` |
| Farbreihenfolge (OpenCV liefert **BGR**) | `kh.bild_vorbereiten()` wird in Training **und** Betrieb verwendet |
| Normierung (÷ 255 bzw. −1 … 1) | steckt als `Rescaling`-Schicht **im** Modell |
| Reihenfolge der Klassen | steht in `modell_info.json` (aus `klassen.txt`) |
| Beleuchtung, Abstand, Hintergrund | **nur durch den Aufbau der Station** – deshalb fest montierte Kamera und eigene Lichtquelle |

**TensorFlow Lite** ist eine schlanke Laufzeitumgebung für Kleinstrechner. Sie lädt schneller, braucht weniger RAM und rechnet auf dem Raspberry Pi meist deutlich schneller als Keras.

### Gespeicherte Modelle

In [ ]:
infos = []
for datei in sorted((ORDNER / "modelle").glob("*_info.json")):
    info = json.loads(datei.read_text(encoding="utf-8"))
    info["info_datei"] = datei.name
    infos.append(info)

if not infos:
    print("Noch keine Modelle gespeichert – bitte zuerst Notebook 11, 12 oder 13 ausführen.")
pd.DataFrame(infos).reindex(columns=["name", "klassen", "groesse", "farbe", "test_genauigkeit", "erstellt"]) if infos else None

### 🔍 Modelle vergleichen: Genauigkeit, Rechenzeit, Größe
Gemessen wird die Zeit für **eine** Vorhersage inklusive Bildvorbereitung – so wie später an der Station.

In [ ]:
beispiel_bgr = cv2.imread(str(kh.bilddateien(ORDNER / "dataset" / "0")[0]))
vergleich = []
for info in infos:
    for tflite in (True, False):
        if tflite and "tflite" not in info:
            continue
        k = kh.Klassifikator(ORDNER / "modelle" / info["info_datei"], bevorzuge_tflite=tflite)
        ms = kh.messe_inferenzzeit(k.vorhersage, beispiel_bgr, wiederholungen=20)
        datei = ORDNER / "modelle" / (info["tflite"] if tflite else info["keras"])
        vergleich.append({"Modell": info["name"], "Laufzeit": k.art, "Bild": f'{info["groesse"]} {info["farbe"]}',
                          "Test-Genauigkeit": info.get("test_genauigkeit"),
                          "ms pro Bild": round(ms, 1), "Datei [MB]": round(datei.stat().st_size / 1e6, 2)})
        del k
pd.DataFrame(vergleich)

🔍 **Entscheidet im Team** und begründet schriftlich: Welches Modell setzt ihr an der Prüfstation ein? Berücksichtigt Genauigkeit, Rechenzeit (Taktzeit!) und Robustheit (Grad-CAM aus Notebook 13).

In [ ]:
MODELL = "cnn"                 # "fcnn", "cnn" oder "transfer"
KAMERA = 0                     # 0 = USB-Kamera, "picamera" = Kameramodul
SCHWELLE = 0.80                # Mindest-Konfidenz für eine automatische Entscheidung

klassifikator = kh.Klassifikator(ORDNER / "modelle" / f"{MODELL}_info.json")
klassen = klassifikator.klassen
print(f"Modell {MODELL} ({klassifikator.art}) · Klassen {klassen} · Bild {klassifikator.groesse} {klassifikator.farbe}")

### ✏️ Aufgabe A1 – Die Entscheidungslogik
Schreibe `entscheide(wahrsch, klassen, schwelle)`. Die Funktion bekommt die Softmax-Ausgabe und gibt drei Werte zurück:
- `ergebnis`: Name der wahrscheinlichsten Klasse – oder `"unsicher"`, wenn deren Wahrscheinlichkeit kleiner als `schwelle` ist,
- `konfidenz`: die höchste Wahrscheinlichkeit,
- `nummer`: die Klassennummer (auch wenn unsicher).

In [ ]:
def entscheide(wahrsch, klassen, schwelle):
    return "unsicher", 0.0, -1   # TODO


k4 = ["Kreuz", "Kreis", "Dreieck", "Quadrat"]
kh.pruefe(lambda: entscheide(np.array([0.05, 0.9, 0.03, 0.02]), k4, 0.8)[0] == "Kreis",
          "Sichere Entscheidung richtig", "[0.05, 0.9, 0.03, 0.02] mit Schwelle 0.8 → 'Kreis'")
kh.pruefe(lambda: entscheide(np.array([0.4, 0.1, 0.45, 0.05]), k4, 0.8)[0] == "unsicher",
          "Unsichere Entscheidung erkannt", "[0.4, 0.1, 0.45, 0.05] mit Schwelle 0.8 → 'unsicher'")
kh.pruefe(lambda: entscheide(np.array([0.4, 0.1, 0.45, 0.05]), k4, 0.8)[1:] == (0.45, 2),
          "Konfidenz und Klassennummer richtig", "Erwartet: konfidenz 0.45 und nummer 2")

### Test mit gespeicherten Bildern
Zuerst ohne Kamera: zufällige Bilder aus dem Datensatz – so, wie sie von der Kamera kämen (volle Auflösung, BGR).

⚠️ Diese Bilder hat das Modell zum Teil im Training gesehen. Einen ehrlichen Test liefern nur **neue** Aufnahmen – die macht ihr gleich live.

In [ ]:
rng = np.random.default_rng()
plt.figure(figsize=(16, 6))
for n in range(8):
    klasse = rng.integers(len(klassen))
    dateien = kh.bilddateien(ORDNER / "dataset" / str(klasse))
    bild = cv2.imread(str(dateien[rng.integers(len(dateien))]))
    ergebnis, konf, _ = entscheide(klassifikator.vorhersage(bild), klassen, SCHWELLE)
    plt.subplot(2, 4, n + 1); plt.imshow(bild[..., ::-1]); plt.axis("off")
    farbe = "orange" if ergebnis == "unsicher" else ("green" if ergebnis == klassen[klasse] else "red")
    plt.title(f"echt: {klassen[klasse]}\n→ {ergebnis} ({konf:.0%})", color=farbe)
plt.tight_layout(); plt.show()

### Live-Erkennung im Notebook
Die Zelle zeigt für `SEKUNDEN` Sekunden das Kamerabild mit der Entscheidung an. Halte verschiedene Symbole – auch neue, selbst gezeichnete – unter die Kamera.

🍓 Für den Dauerbetrieb mit flüssigem Bild nutzt ihr später das Skript `20_Symbolerkennung_live.py` (siehe unten). Abbruch hier mit ■ (Interrupt).

In [ ]:
from IPython.display import Image, display

def zeichne_ergebnis(bild, ergebnis, konfidenz):
    farbe = (0, 165, 255) if ergebnis == "unsicher" else (0, 170, 0)
    cv2.rectangle(bild, (0, 0), (bild.shape[1], 50), (0, 0, 0), -1)
    cv2.putText(bild, f"{ergebnis}  {konfidenz:.0%}", (12, 36), cv2.FONT_HERSHEY_SIMPLEX, 1.1, farbe, 2)
    return bild

SEKUNDEN = 30
anzeige = display(None, display_id=True)
try:
    with kh.Kamera(KAMERA, 640, 480) as kamera:
        ende = time.time() + SEKUNDEN
        while time.time() < ende:
            bild = kamera.lies()
            if bild is None:
                break
            ergebnis, konf, _ = entscheide(klassifikator.vorhersage(bild), klassen, SCHWELLE)
            _, png = cv2.imencode(".jpg", zeichne_ergebnis(bild, ergebnis, konf))
            anzeige.update(Image(data=png.tobytes()))
except KeyboardInterrupt:
    print("beendet")
except RuntimeError as e:
    print("Kamera nicht verfügbar:", e)

### ✏️ Aufgabe A2 – Taktzeit bestimmen
🏭 Die **Taktzeit** gibt an, wie viel Zeit pro Teil zur Verfügung steht. Miss die mittlere Zeit für die **Bildaufnahme** und für die **Inferenz** (jeweils in Millisekunden) über 20 Bilder und berechne, wie viele Teile pro Minute die Station höchstens prüfen kann.

In [ ]:
zeiten_aufnahme, zeiten_inferenz = [], []
try:
    with kh.Kamera(KAMERA, 640, 480) as kamera:
        for _ in range(20):
            t0 = time.perf_counter()
            bild = kamera.lies()
            t1 = time.perf_counter()
            klassifikator.vorhersage(bild)
            t2 = time.perf_counter()
            zeiten_aufnahme.append((t1 - t0) * 1000)
            zeiten_inferenz.append((t2 - t1) * 1000)
except RuntimeError as e:
    print("Kamera nicht verfügbar – es werden Beispielwerte verwendet:", e)
    zeiten_aufnahme, zeiten_inferenz = [33.0] * 20, [kh.messe_inferenzzeit(klassifikator.vorhersage, beispiel_bgr)] * 20

t_aufnahme = None        # TODO: Mittelwert in ms
t_inferenz = None        # TODO: Mittelwert in ms
teile_pro_minute = None  # TODO

print(f"Aufnahme {t_aufnahme:.1f} ms + Inferenz {t_inferenz:.1f} ms → höchstens {teile_pro_minute:.0f} Teile/min")
kh.pruefe(lambda: abs(teile_pro_minute - 60_000 / (np.mean(zeiten_aufnahme) + np.mean(zeiten_inferenz))) < 1e-6,
          "Taktzeit richtig berechnet", "Teile pro Minute = 60 000 ms / (Aufnahmezeit + Inferenzzeit)")

🔍 Das Förderband transportiert 40 Teile pro Minute. Reicht die Station? Wie viel Reserve bleibt? Was würde sich mit dem Transfer-Modell ändern?

### 🏭 Signalausgabe: Ampel am GPIO-Port
Die Station zeigt ihr Ergebnis mit drei LEDs an. Ohne angeschlossene LEDs wird die Ampel nur simuliert.

| LED | GPIO (BCM) | Pin | Bedeutung |
|---|:--:|:--:|---|
| grün | 22 | 15 | Klasse sicher erkannt (z. B. i. O.) |
| gelb | 27 | 13 | unsicher → manuelle Prüfung |
| rot | 17 | 11 | Ausschuss-Klasse erkannt |

Jede LED mit einem Vorwiderstand (220 – 330 Ω) gegen GND (z. B. Pin 9) anschließen.

⚠️ **In der Industrie** steuert der Raspberry Pi keine Aktoren direkt. Er meldet sein Ergebnis an die Steuerung (SPS) – über ein 24-V-Ein-/Ausgabemodul mit Optokoppler oder über ein Netzwerkprotokoll wie OPC UA oder MQTT. **Sicherheitsfunktionen** (Not-Halt, Schutztüren) werden nie von einer KI übernommen.

In [ ]:
class Ampel:
    def __init__(self, rot=17, gelb=27, gruen=22):
        try:
            from gpiozero import LED
            self.leds = {"rot": LED(rot), "gelb": LED(gelb), "gruen": LED(gruen)}
            self.simulation = False
        except Exception as e:                       # kein Pi, keine LEDs, kein gpiozero
            self.leds, self.simulation = {}, True
            print("Ampel wird simuliert:", type(e).__name__)

    def zeige(self, farbe):
        for name, led in self.leds.items():
            led.on() if name == farbe else led.off()
        if self.simulation:
            print({"rot": "🔴", "gelb": "🟡", "gruen": "🟢"}[farbe], end=" ")

    def aus(self):
        for led in self.leds.values():
            led.off()


AUSSCHUSS_KLASSEN = {"Kreuz"}       # welche Klassen bedeuten "rot"? (an eure Klassen anpassen)

def ampelfarbe(ergebnis):
    if ergebnis == "unsicher":
        return "gelb"
    return "rot" if ergebnis in AUSSCHUSS_KLASSEN else "gruen"

ampel = Ampel()
for e in ["Kreis", "unsicher", "Kreuz", "Quadrat"]:
    ampel.zeige(ampelfarbe(e)); time.sleep(0.5)
ampel.aus()

### ✏️ Aufgabe A3 – Prüfprotokoll
🏭 Jede Entscheidung muss **nachvollziehbar** sein (Rückverfolgbarkeit, Qualitätsnachweis). Ergänze `protokolliere()`: Sie soll an die CSV-Datei eine Zeile im Format

`Zeitstempel;Ergebnis;Konfidenz`

anhängen (Konfidenz mit 3 Nachkommastellen). Unsichere Bilder werden zusätzlich gespeichert – sie sind wertvolle neue Trainingsdaten!

In [ ]:
PROTOKOLL = ORDNER / "protokoll" / "pruefprotokoll.csv"
PROTOKOLL.parent.mkdir(exist_ok=True)
(PROTOKOLL.parent / "unsicher").mkdir(exist_ok=True)

def protokolliere(ergebnis, konfidenz, bild=None, datei=PROTOKOLL):
    zeit = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
    zeile = ""   # TODO: Zeitstempel;Ergebnis;Konfidenz
    neu = not datei.exists()
    with open(datei, "a", encoding="utf-8") as f:
        if neu:
            f.write("zeit;ergebnis;konfidenz\n")
        f.write(zeile + "\n")
    if ergebnis == "unsicher" and bild is not None:
        cv2.imwrite(str(datei.parent / "unsicher" / f"{zeit.replace(':', '-').replace(' ', '_')}.png"), bild)


test_datei = PROTOKOLL.parent / "test.csv"
test_datei.unlink(missing_ok=True)
protokolliere("Kreis", 0.91234, datei=test_datei)
letzte = test_datei.read_text(encoding="utf-8").strip().splitlines()[-1]
print(letzte)
kh.pruefe(lambda: letzte.split(";")[1:] == ["Kreis", "0.912"], "Protokollzeile richtig",
          "Erwartet: <Zeit>;Kreis;0.912")
test_datei.unlink(missing_ok=True)

### Dauerbetrieb mit dem Skript 20
Das Skript `20_Symbolerkennung_live.py` vereint alles: Kamera, Modell, Konfidenzschwelle, Ampel und Protokoll. Start im Terminal:

```bash
python3 20_Symbolerkennung_live.py --modell cnn --schwelle 0.8
python3 20_Symbolerkennung_live.py --modell transfer --kamera picamera --gpio --ausschuss Kreuz
```

Tasten: **ESC/q** beenden · **p** Pause · **+ / −** Schwelle ändern · **s** aktuelles Bild speichern.

## 🧪 Teilzielkontrolle 4

In [ ]:
kh.zeige_html("tzk4_transfer_und_einsatz.html", hoehe=900)

---
## Teil B – 🏭 Übertrag in die industrielle Fertigung (Doppelstunde 10)

Die Symbole waren ein Stellvertreter. Dieselbe Pipeline löst viele echte Prüfaufgaben – es ändern sich nur **die Klassen und die Bilder**:

| Anwendung | mögliche Klassen | Besonderheit |
|---|---|---|
| **Teilesortierung / Variantenkontrolle** | Schraube M6 · Schraube M8 · Mutter · Scheibe · leer | ähnliche Teile → höhere Auflösung, feste Lage |
| **Oberflächenprüfung** | i. O. · Kratzer · Delle · Verschmutzung | Fehlerbilder sind selten → gezielt sammeln, Streiflicht |
| **Vollständigkeitsprüfung** | vollständig · Bauteil fehlt · falsches Bauteil | Kamera exakt über der Baugruppe |
| **Lageerkennung** (Roboter-Zuführung) | richtig · um 180° gedreht · auf der Seite | **keine** Dreh-/Spiegel-Augmentierung! |
| **Kennzeichnung / Etiketten** | Etikett vorhanden · fehlt · schief | |
| **Werkzeugzustand** | Fräser neu · verschlissen · gebrochen | Bilder in festen Abständen aufnehmen |
| **Prüfmarken** (wie unsere Symbole) | QS-Stempel „i. O.“ · „Nacharbeit“ · „Sperre“ | |

### Was beim echten Einsatz zusätzlich wichtig ist
- **Aufbau:** fest montierte Kamera, eigene Beleuchtung (Ringlicht/Durchlicht), Abschirmung gegen Tageslicht, einheitlicher Hintergrund.
- **Auslösung:** Bild genau dann aufnehmen, wenn das Teil an der richtigen Stelle ist (Lichtschranke/Sensor statt Dauerbetrieb).
- **Klasse „leer“** und eine **Rückweisung** bei niedriger Konfidenz einplanen.
- **Fehlerkosten** bestimmen die Schwelle (Schlupf vs. Pseudoausschuss, Notebook 12).
- **Datendrift überwachen:** neue Lieferanten, Verschleiß der Lampe, verschmutzte Linse → Genauigkeit regelmäßig mit neuen Stichproben prüfen und nachtrainieren.
- **Dokumentation:** verwendeter Datensatz, Modellversion, Testergebnisse, Schwelle – damit jede Entscheidung nachvollziehbar ist.
- **Mensch im Prozess:** Unsichere Teile prüft ein Mensch; seine Entscheidung wird als neues Trainingsbild gespeichert (Datenkreislauf).

### ✏️ Projektauftrag
Wählt im Team eine Anwendung aus der Tabelle (oder eine eigene aus eurem Betrieb) und setzt die komplette Pipeline um:

1. **Planen:** Klassen festlegen, Aufbau skizzieren (Kamera, Licht, Hintergrund), Fehlerkosten abschätzen.
2. **Aufnehmen:** `python3 00_Datensatz_aufnehmen.py --datensatz projekt_<team> --klassen …` – mindestens 80 Bilder je Klasse.
3. **Trainieren:** In Notebook 12 oder 13 `DATENSATZ = ORDNER / "projekt_<team>"` setzen und alle Zellen ausführen.
4. **Bewerten:** Konfusionsmatrix, Grad-CAM, Schwelle festlegen.
5. **Einsetzen:** `python3 20_Symbolerkennung_live.py --modell <name>` mit **neuen** Teilen testen.
6. **Präsentieren** (5 Minuten): Aufgabe, Daten, Modell, Ergebnis, Grenzen und Verbesserungsideen.

| Kriterium | Gewicht |
|---|:--:|
| Datensatz: Umfang, Ausgewogenheit, Vielfalt, passende Klassen (inkl. „leer“) | 25 % |
| Modell: begründete Wahl, Lernkurven gedeutet, Testergebnis | 25 % |
| Bewertung: Konfusionsmatrix, Schwelle mit Fehlerkosten begründet | 20 % |
| Live-Test an der Station mit neuen Teilen | 15 % |
| Präsentation: Grenzen und Risiken realistisch eingeschätzt | 15 % |

### Reflexion
1. Was war die wichtigste Stellschraube für ein gutes Ergebnis – das Modell oder die Daten?
2. In welchen Situationen würdet ihr der Prüfstation **nicht** vertrauen?
3. Was würde sich ändern, wenn statt 4 Klassen 40 verschiedene Bauteile erkannt werden müssten?